# Coffee17 Preprocessing KD — Exploratory v1\n\nNotebook ini **tidak mengulang 20 primary runs**. Ia membaca saved output preprocessing lama sebagai Kaggle Input, memverifikasi split/hash, lalu menjalankan hanya dua treatment student: **KD-R0** dan **KD-ALL4** pada 5 fold.\n\nSebelum **Save & Run All**:\n1. Add Input: dataset **Coffee Green Bean with 17 Defects Original**.\n2. Add Input: **saved output version** dari notebook preprocessing primary yang berisi `coffee17-preprocessing-project`.\n3. Aktifkan GPU dan Internet.\n4. Run All.\n\nOutput persisten hanya `coffee17-preprocessing-kd-project`; dataset/canonical clone temporer dibersihkan sebelum notebook selesai.\n

In [ ]:
# Coffee17 preprocessing KD — saved-primary -> KD-R0 / KD-ALL4
SCIENTIFIC_CODE_COMMIT = "0e979d191cbc864099b4ff8e0dd0795dca95dafa"

import gc, hashlib, importlib, json, os, shutil, subprocess, sys, zipfile
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
REPO = WORK / "coffee-bean-classification-kd-code"
KD_PROJECT = WORK / "coffee17-preprocessing-kd-project"

assert INPUT.is_dir() and WORK.is_dir(), "Notebook ini harus dijalankan di Kaggle."

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def merge_tree_exact(source, target):
    source, target = Path(source), Path(target)
    for item in sorted(source.rglob("*")):
        if not item.is_file():
            continue
        rel = item.relative_to(source)
        dst = target / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        if dst.is_file():
            if sha256_file(item) != sha256_file(dst):
                raise RuntimeError(f"Kaggle input conflict: {rel}")
        else:
            shutil.copy2(item, dst)

def run(command, cwd=None, log_path=None):
    command = [str(x) for x in command]
    print("\n$ " + " ".join(command), flush=True)
    if log_path is None:
        subprocess.run(command, cwd=cwd, check=True)
        return
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("a", encoding="utf-8") as stream:
        process = subprocess.Popen(
            command,
            cwd=cwd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            stream.write(line)
            stream.flush()
        rc = process.wait()
    if rc:
        raise RuntimeError(f"Command gagal ({rc}): {' '.join(command)}")

# ------------------------------------------------------------------
# 1. Locate the already-saved primary preprocessing output.
# ------------------------------------------------------------------
primary_candidates = []
for p in INPUT.rglob("coffee17-preprocessing-project"):
    if not p.is_dir():
        continue
    authority = p / "evidence/coffee17-preprocessing-primary-v1/preprocessing_primary_confirmation.json"
    experiments = p / "experiments/coffee17-preprocessing-primary-v1"
    clean_manifest = p / "evidence/coffee17-preprocessing-data-v1/clean_manifest.json"
    fold_manifest = p / "evidence/coffee17-preprocessing-data-v1/fold_manifest.json"
    if all(x.is_file() for x in (authority, clean_manifest, fold_manifest)) and experiments.is_dir():
        payload = json.loads(authority.read_text(encoding="utf-8"))
        if payload.get("decision") == "AUTHORIZE_OOF_TEST_EVALUATION" and payload.get("completed_runs") == 20:
            primary_candidates.append(p)

if not primary_candidates:
    raise FileNotFoundError(
        "Saved output coffee17-preprocessing-project tidak ditemukan. "
        "Add Input -> pilih output version preprocessing lama."
    )

authority_hashes = {
    sha256_file(
        p / "evidence/coffee17-preprocessing-primary-v1/preprocessing_primary_confirmation.json"
    )
    for p in primary_candidates
}
if len(authority_hashes) != 1:
    raise RuntimeError(
        "Ada beberapa primary output dengan authority berbeda. "
        "Attach hanya version primary yang benar."
    )

PRIMARY_PROJECT = sorted(primary_candidates, key=lambda p: str(p))[0]
print("PRIMARY PROJECT:", PRIMARY_PROJECT)

DATA_EVIDENCE = PRIMARY_PROJECT / "evidence/coffee17-preprocessing-data-v1"
AUTHORITY = PRIMARY_PROJECT / "evidence/coffee17-preprocessing-primary-v1/preprocessing_primary_confirmation.json"
PRIMARY_EXPERIMENTS = PRIMARY_PROJECT / "experiments/coffee17-preprocessing-primary-v1"

# Optional resume from a previous saved KD notebook output.
for prior in sorted(
    p for p in INPUT.rglob("coffee17-preprocessing-kd-project") if p.is_dir()
):
    print("MERGE PRIOR KD OUTPUT:", prior)
    merge_tree_exact(prior, KD_PROJECT)

KD_PROJECT.mkdir(parents=True, exist_ok=True)
(KD_PROJECT / "audits").mkdir(parents=True, exist_ok=True)
(KD_PROJECT / "logs").mkdir(parents=True, exist_ok=True)
(KD_PROJECT / "analysis").mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# 2. Clone the exact scientific code and install the frozen environment.
# ------------------------------------------------------------------
if REPO.exists():
    shutil.rmtree(REPO)
run([
    "git", "clone", "--quiet", "--no-checkout",
    "https://github.com/ediprin/coffee-bean-classification.git", REPO
])
run(["git", "-C", REPO, "checkout", "--quiet", "--detach", SCIENTIFIC_CODE_COMMIT])

prior_lock = PRIMARY_PROJECT / "evidence/coffee17-preprocessing-runtime-v1/requirements_preprocessing_study_lock.txt"
requirements = prior_lock if prior_lock.is_file() else REPO / "requirements/preprocessing-study.txt"
run([sys.executable, "-m", "pip", "install", "-q", "-r", requirements])
run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", REPO])
sys.path.insert(0, str(REPO / "src"))
importlib.invalidate_caches()
os.chdir(REPO)

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Aktifkan Kaggle GPU sebelum Run All.")
print("GPU:", torch.cuda.get_device_name(0))

from bilinear_lmmd.data.preparation.prepare_coffee17 import discover_directory_samples
from bilinear_lmmd.data.preparation.audit_coffee17_provenance import audit_coffee17_provenance
from bilinear_lmmd.data.preparation.prepare_preprocessing_folds import prepare_preprocessing_folds
from bilinear_lmmd.data.preparation.materialize_preprocessing_development import (
    materialize_preprocessing_development,
)

# ------------------------------------------------------------------
# 3. Reconstruct Coffee17 locally, then prove the split is identical
#    to the frozen primary evidence. No primary training is repeated.
# ------------------------------------------------------------------
print("\n=== RECONSTRUCT + VERIFY COFFEE17 ===")
by_class = discover_directory_samples(INPUT)
raw_count = sum(len(v) for v in by_class.values())
print("Coffee17 mounted:", raw_count, "images /", len(by_class), "classes")
if raw_count != 979 or len(by_class) != 17:
    raise RuntimeError(
        f"Coffee17 input tidak sesuai expected raw population: {raw_count}/17."
    )

ARCHIVE = WORK / "coffee17_kd_original.zip"
PROV_LOCAL = WORK / "coffee17_kd_provenance"
CANONICAL = WORK / "coffee17_kd_original_v1"
FOLDS_LOCAL = WORK / "coffee17_kd_folds"

for path in (PROV_LOCAL, CANONICAL, FOLDS_LOCAL):
    shutil.rmtree(path, ignore_errors=True)
if ARCHIVE.exists():
    ARCHIVE.unlink()

with zipfile.ZipFile(ARCHIVE, "w", compression=zipfile.ZIP_STORED) as bundle:
    for class_name, paths in sorted(by_class.items()):
        for path in sorted(paths):
            info = zipfile.ZipInfo(
                f"{class_name}/{path.name}",
                date_time=(1980, 1, 1, 0, 0, 0),
            )
            info.compress_type = zipfile.ZIP_STORED
            info.external_attr = 0o644 << 16
            bundle.writestr(info, path.read_bytes())

provenance = audit_coffee17_provenance(
    ARCHIVE, PROV_LOCAL, canonical_root=CANONICAL
)
if provenance["decision"] != "PASS":
    raise RuntimeError(f"Provenance gagal: {provenance['decision']}")

fold_summary = prepare_preprocessing_folds(
    CANONICAL,
    PROV_LOCAL / "coffee17_provenance.json",
    FOLDS_LOCAL,
    folds=5,
    seed=42,
    validation_ratio=0.10,
)
if fold_summary["decision"] != "PASS_COFFEE17_PREPROCESSING_DATA_GATE":
    raise RuntimeError("Fold reconstruction gagal.")

for name in ("clean_manifest.json", "fold_manifest.json"):
    reconstructed = FOLDS_LOCAL / name
    frozen = DATA_EVIDENCE / name
    if sha256_file(reconstructed) != sha256_file(frozen):
        raise RuntimeError(
            f"Reconstructed {name} berbeda dari frozen primary evidence."
        )
print("DATA/FOLD HASH MATCH: PASS")

# ------------------------------------------------------------------
# 4. Diagnostic teacher audit + two frozen KD treatments.
# ------------------------------------------------------------------
KD_OUT = KD_PROJECT / "experiments/coffee17-preprocessing-kd-exploratory-v1"
CONFIGS = {
    "R0": REPO / "configs/preprocessing_kd/KD_R0.yaml",
    "ALL4": REPO / "configs/preprocessing_kd/KD_ALL4.yaml",
}

for fold in range(1, 6):
    print(f"\n================ FOLD {fold}/5 ================", flush=True)
    DEV = WORK / f"coffee17_kd_dev_fold_{fold}"
    shutil.rmtree(DEV, ignore_errors=True)
    materialize_preprocessing_development(
        CANONICAL,
        DATA_EVIDENCE / "clean_manifest.json",
        DATA_EVIDENCE / "fold_manifest.json",
        DEV,
        fold=fold,
    )

    audit_path = KD_PROJECT / "audits" / f"teacher_audit_fold{fold}.json"
    if not audit_path.is_file():
        run([
            sys.executable, "-u", "-m",
            "bilinear_lmmd.experiments.run_preprocessing_kd_teacher_audit",
            "--data-root", DEV,
            "--authority", AUTHORITY,
            "--experiments-root", PRIMARY_EXPERIMENTS,
            "--fold", str(fold),
            "--output", audit_path,
            "--device", "cuda:0",
        ], cwd=REPO, log_path=KD_PROJECT / "logs" / f"audit_fold{fold}.log")
    else:
        print("REUSE AUDIT:", audit_path)

    gc.collect()
    torch.cuda.empty_cache()

    for mode in ("R0", "ALL4"):
        print(f"\n--- KD {mode} / fold {fold} ---", flush=True)
        run([
            sys.executable, "-u", "-m",
            "bilinear_lmmd.experiments.run_preprocessing_kd_arm",
            "--config", CONFIGS[mode],
            "--fold", str(fold),
            "--data-root", DEV,
            "--authority", AUTHORITY,
            "--experiments-root", PRIMARY_EXPERIMENTS,
            "--output-root", KD_OUT,
            "--required-commit", SCIENTIFIC_CODE_COMMIT,
            "--device", "cuda:0",
            "--resume",
            "--authorize-training",
        ], cwd=REPO, log_path=KD_PROJECT / "logs" / f"KD_{mode}_fold{fold}.log")

        metrics_path = KD_OUT / mode / f"fold_{fold}" / "seed42" / "validation" / "metrics.json"
        metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
        print(
            f"KD {mode} fold {fold} DONE | "
            f"Macro-F1={metrics['macro_f1']:.4f} | "
            f"Worst-F1={metrics['worst_class_f1']:.4f}"
        )
        gc.collect()
        torch.cuda.empty_cache()

    shutil.rmtree(DEV, ignore_errors=True)

# ------------------------------------------------------------------
# 5. Descriptive validation summary. No outer OOF/test is opened here.
# ------------------------------------------------------------------
SUMMARY = KD_PROJECT / "analysis" / "kd_validation_summary.json"
run([
    sys.executable, "-u", "-m",
    "bilinear_lmmd.experiments.run_preprocessing_kd_validation_summary",
    "--experiments-root", PRIMARY_EXPERIMENTS,
    "--kd-root", KD_OUT,
    "--output", SUMMARY,
], cwd=REPO)

print("\n=== KD EXPLORATORY RUN COMPLETE ===")
print("SUMMARY:", SUMMARY)
print("OUTPUT :", KD_PROJECT)

# ------------------------------------------------------------------
# 6. Cleanup large temporary copies so Save Version stores KD evidence only.
# ------------------------------------------------------------------
for path in (REPO, PROV_LOCAL, CANONICAL, FOLDS_LOCAL):
    shutil.rmtree(path, ignore_errors=True)
if ARCHIVE.exists():
    ARCHIVE.unlink()

total_bytes = sum(p.stat().st_size for p in KD_PROJECT.rglob("*") if p.is_file())
print("KD PROJECT SIZE:", round(total_bytes / 1024 / 1024, 2), "MB")
print("Gunakan Save Version. Tidak ada primary R0/C0/F0/W0 training yang diulang.")
